# Notebook 2: Train & Evaluate a Text Classifier

In the previous notebook, we explored the AG News dataset and saved it for reuse. Now it's time to build a model.

**What you'll do in this notebook:**

1. Load the data we saved in notebook 1
2. Create a train/validation split
3. Build a TF-IDF + Logistic Regression pipeline
4. Train the model and evaluate its performance
5. Track everything with MLflow

**What you'll learn:**
- How TF-IDF converts raw text into numbers a model can understand
- How Logistic Regression works as a simple but effective classifier
- Why we split data into train/validation/test sets
- How to read a confusion matrix and classification report
- How MLflow helps you organize and compare experiments

---

### Notebook series

| Notebook | Focus |
|---|---|
| 01 — Setup & EDA | Environment, data loading, exploration |
| **02 — Train & Evaluate** (you are here) | Build, train, and evaluate a text classifier |
| 03 — Serve & Predict | Load the trained model and make predictions |

---
## 1. Setup

First, let's import our libraries and reload the data from notebook 1.

In [31]:
import os, sys, yaml
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

import mlflow
import mlflow.sklearn

# Locate the project root
PROJECT_ROOT = (
    Path("__vsc_ipynb_file__").resolve().parent.parent
    if "__vsc_ipynb_file__" in dir()
    else Path.cwd().parent
)
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import set_all_seeds, get_env

set_all_seeds(42)
print(f"Project root: {PROJECT_ROOT}")
print("Random seed set to 42.")

Project root: /Users/Bartley/Documents/personal_dev/h4la/repos/data-science/tutorials/setup_template
Random seed set to 42.


### Load saved data and config

In [32]:
# Load config
cfg_path = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Load DataFrames saved by notebook 1
data_dir = PROJECT_ROOT / "data" / "interim"
train_df = pd.read_parquet(data_dir / "train.parquet")
test_df  = pd.read_parquet(data_dir / "test.parquet")
valid_df = pd.read_parquet(data_dir / "valid.parquet") if (data_dir / "valid.parquet").exists() else None

print(f"Train: {len(train_df):,} rows")
print(f"Test:  {len(test_df):,} rows")
if valid_df is not None:
    print(f"Valid: {len(valid_df):,} rows")
else:
    print("Valid: None (will create below)")

Train: 120,000 rows
Test:  7,600 rows
Valid: None (will create below)


---
## 2. Create a Train/Validation Split

AG News doesn't come with a built-in validation set, so we need to create one by setting aside some of the training data.

### Why do we need a validation set?

The **test set** should only be used once to get an honest performance estimate. But during development, you often want to check how well your model is doing (e.g., to try different hyperparameters). That's what the **validation set** is for: a "practice test" you can use as often as you like without contaminating your final evaluation.

### What is stratified sampling?

When we split the data, we use `stratify=train_df["label"]`. This ensures each split has the same proportion of each class as the original. Without stratification, random chance might give us a validation set with very few "Sports" articles, for example, which would make our validation metrics unreliable.

In [33]:
if valid_df is None:
    train_df, valid_df = train_test_split(
        train_df,
        test_size=cfg["test_size"],      # 20% goes to validation, a good rule of thumb is 60% train - 20% validation 20% test
        random_state=cfg["random_state"], # reproducible split
        stratify=train_df["label"]        # keep class proportions balanced
    )
    print(f"Split training data into:")
    print(f"  Train:      {len(train_df):,} rows (80%)")
    print(f"  Validation: {len(valid_df):,} rows (20%)")
else:
    print(f"Using existing validation split ({len(valid_df):,} rows)")

# Separate features (X) from labels (y) — standard ML convention
X_train, y_train = train_df["text"].astype(str), train_df["label"]
X_valid, y_valid = valid_df["text"].astype(str), valid_df["label"]
X_test,  y_test  = test_df["text"].astype(str),  test_df["label"]

print(f"\nFinal sizes:")
print(f"  Train:      {len(X_train):,}")
print(f"  Validation: {len(X_valid):,}")
print(f"  Test:       {len(X_test):,}")

Split training data into:
  Train:      96,000 rows (80%)
  Validation: 24,000 rows (20%)

Final sizes:
  Train:      96,000
  Validation: 24,000
  Test:       7,600


---
## 3. Understand the Model: TF-IDF + Logistic Regression

Before we build the model, let's understand what it actually does. Our pipeline has two stages:

### Stage 1: TF-IDF (turning text into numbers)

Machine learning models work with numbers, not raw text. **TF-IDF** (Term Frequency - Inverse Document Frequency) converts each article into a vector of numbers by:

1. **Term Frequency (TF):** How often does each word appear in *this* article?
2. **Inverse Document Frequency (IDF):** How rare is this word across *all* articles?

Words that appear frequently in one article but rarely overall get high TF-IDF scores. Common words like "the" and "is" get low scores because they appear everywhere and don't help distinguish categories.

**Key settings from our config:**
- `max_features=30000` — Only keep the 30,000 most informative terms. This reduces noise and speeds up training.
- `ngram_range=[1, 2]` — Consider both single words ("market") and two-word phrases ("stock market"). Bigrams capture useful context that single words miss.

### Stage 2: Logistic Regression (making the prediction)

Despite the name, **Logistic Regression** is a *classification* algorithm (not regression). It learns a set of weights — one per feature — and uses them to compute the probability that an article belongs to each class. The class with the highest probability wins.

**Why start with Logistic Regression?**
- It's fast to train (seconds, not hours)
- It's interpretable (you can look at which words matter most)
- It's a strong baseline — often surprisingly competitive
- It establishes a performance floor before trying more complex models

**Key settings:**
- `C=2.0` — Controls regularization. Higher C = less regularization = the model fits the training data more closely. Too high and it overfits; too low and it underfits.
- `max_iter=200` — Maximum number of optimization steps. If the model hasn't converged by 200 iterations, something might be wrong.

### Build the sklearn Pipeline

Scikit-learn's `Pipeline` chains the two steps together so they act as a single unit. This is convenient because:
- Calling `pipe.fit(X, y)` runs TF-IDF fitting *and* model training in one step
- Calling `pipe.predict(X)` runs TF-IDF transformation *and* prediction
- You can't accidentally forget to transform your data before predicting

In [34]:
pipe = Pipeline([
    # Step 1: Convert text -> TF-IDF feature vectors
    ("tfidf", TfidfVectorizer(
        max_features=cfg["tfidf"]["max_features"],
        ngram_range=tuple(cfg["tfidf"]["ngram_range"])
    )),
    # Step 2: Classify the feature vectors
    ("clf", LogisticRegression(
        C=cfg["model"]["C"],
        max_iter=cfg["model"]["max_iter"]
    ))
])

print("Pipeline structure:")
print(pipe)

Pipeline structure:
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=30000, ngram_range=(1, 2))),
                ('clf', LogisticRegression(C=2.0, max_iter=200))])


---
## 4. Train the Model with MLflow Tracking

### What is MLflow?

[MLflow](https://mlflow.org/) is an experiment tracking tool. Every time you train a model, it records:
- **Parameters** — the settings you used (C, max_features, etc.)
- **Metrics** — how well the model performed (accuracy, F1 score)
- **Artifacts** — files like the saved model, plots, and reports

This is invaluable when you're experimenting. Instead of trying to remember "which run had C=2.0 with bigrams?", MLflow keeps a log of everything.

### Understanding the metrics

- **Accuracy** — What percentage of predictions were correct? Simple and intuitive, but can be misleading if classes are imbalanced.
- **F1 Score (macro)** — The harmonic mean of precision and recall, averaged equally across all classes. This gives a balanced view of performance even if some classes are harder than others.
  - **Precision** = Of all articles I predicted as "Sports", how many actually were?
  - **Recall** = Of all actual "Sports" articles, how many did I find?

In [35]:
# Point MLflow at our local tracking directory
tracking_uri = get_env("MLFLOW_TRACKING_URI", "./mlruns")

tracking_dir = PROJECT_ROOT / "mlruns"
tracking_dir.mkdir(exist_ok=True)

mlflow.set_tracking_uri(str(PROJECT_ROOT / tracking_uri))
mlflow.set_experiment(cfg["experiment_name"])

print(f"MLflow tracking URI: {PROJECT_ROOT / tracking_uri}")
print(f"Experiment name: {cfg['experiment_name']}")

Exception: Invalid parent directory '/Users/Bartley/Documents/personal_dev/h4la/repos/data-science/tutorials/setup_template/mlruns/.trash'

In [ ]:
with mlflow.start_run() as run:
    # --- Log parameters so we can look them up later ---
    mlflow.log_params({
        "dataset": cfg["dataset"],
        "tfidf_max_features": cfg["tfidf"]["max_features"],
        "tfidf_ngram_range": str(cfg["tfidf"]["ngram_range"]),
        "model": cfg["model"]["type"],
        "C": cfg["model"]["C"],
        "max_iter": cfg["model"]["max_iter"],
        "random_state": cfg["random_state"]
    })

    # --- Train the model ---
    print("Training... (this may take a minute on the full dataset)")
    pipe.fit(X_train, y_train)
    print("Training complete!\n")

    # --- Generate predictions on validation and test sets ---
    y_pred_valid = pipe.predict(X_valid)
    y_pred_test  = pipe.predict(X_test)

    # --- Compute metrics ---
    avg = cfg["metrics"]["average"]
    metrics = {
        "valid_accuracy":  accuracy_score(y_valid, y_pred_valid),
        "valid_f1_macro":  f1_score(y_valid, y_pred_valid, average=avg),
        "test_accuracy":   accuracy_score(y_test, y_pred_test),
        "test_f1_macro":   f1_score(y_test, y_pred_test, average=avg),
    }
    mlflow.log_metrics(metrics)

    print("Results:")
    print(f"  Validation — Accuracy: {metrics['valid_accuracy']:.4f}, F1 (macro): {metrics['valid_f1_macro']:.4f}")
    print(f"  Test       — Accuracy: {metrics['test_accuracy']:.4f}, F1 (macro): {metrics['test_f1_macro']:.4f}")

    # --- Save the model as an MLflow artifact ---
    mlflow.sklearn.log_model(pipe, artifact_path="model")

    # Keep the run ID — we'll need it to load the model in notebook 3
    RUN_ID = run.info.run_id
    print(f"\nMLflow run ID: {RUN_ID}")
    print("Model saved as MLflow artifact.")

---
## 5. Evaluate the Model

Overall accuracy tells you *how often* the model is right, but not *where* it goes wrong. That's what the confusion matrix and classification report are for.

### 5a. Confusion Matrix

A confusion matrix is a grid that shows what the model predicted vs. what the actual label was. The diagonal shows correct predictions; off-diagonal cells show mistakes.

**How to read it:**
- Each **row** is an actual class (what the article really is)
- Each **column** is a predicted class (what the model guessed)
- A perfect model would have numbers only on the diagonal
- Off-diagonal numbers tell you which classes get confused with each other (e.g., "World" articles misclassified as "Business")

In [ ]:
reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test,
    display_labels=["World", "Sports", "Business", "Sci/Tech"],
    ax=ax,
    cmap="Blues"
)
ax.set_title("Confusion Matrix (Test Set)")

fig.savefig(reports_dir / "confusion_matrix.png", dpi=180, bbox_inches="tight")
print(f"Saved: {reports_dir / 'confusion_matrix.png'}")
plt.show()

### 5b. Classification Report

The classification report gives you per-class precision, recall, and F1 score — a more detailed view than the single F1 number above.

**Reminder of what these mean:**

| Metric | Question it answers |
|---|---|
| **Precision** | When the model says "Sports", how often is it right? |
| **Recall** | Of all actual Sports articles, what fraction did the model find? |
| **F1** | The balance between precision and recall (harmonic mean) |
| **Support** | How many test examples belong to this class |

If a class has high precision but low recall, the model is cautious — it only predicts that class when it's very confident, but misses many actual examples. The reverse (low precision, high recall) means the model over-predicts that class.

In [ ]:
label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
target_names = [label_names[i] for i in sorted(label_names.keys())]

report = classification_report(y_test, y_pred_test, target_names=target_names)
print("Classification Report (Test Set):")
print(report)

# Save the report as a text file
report_path = reports_dir / "classification_report.txt"
report_path.write_text(report)
print(f"Saved: {report_path}")

### 5c. Log Evaluation Artifacts to MLflow

Let's add the confusion matrix and classification report to our MLflow run so they're stored alongside the model.

In [ ]:
# Re-open the same run to add artifacts
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact(str(reports_dir / "confusion_matrix.png"))
    mlflow.log_artifact(str(reports_dir / "classification_report.txt"))
    print("Logged evaluation artifacts to MLflow.")

---
## 6. Explore MLflow Artifacts

Everything we logged is now stored under `mlruns/`. The model artifact includes:

| File | Purpose |
|---|---|
| `model/model.pkl` | The serialized sklearn pipeline (TF-IDF + LogReg) |
| `model/MLmodel` | Metadata about the model (what flavor, how to load it) |
| `model/conda.yaml` | Conda environment for reproducibility |
| `model/requirements.txt` | Pip requirements for the model |

You can explore these visually by launching the MLflow UI:

```bash
mlflow ui --backend-store-uri ./mlruns
```

Then open http://localhost:5000 in your browser.

---
## 7. Save the Run ID for Notebook 3

Notebook 3 needs to know which MLflow run contains our trained model. We'll save the run ID to a small file.

In [ ]:
run_id_path = PROJECT_ROOT / "data" / "interim" / "latest_run_id.txt"
run_id_path.write_text(RUN_ID)
print(f"Saved run ID to: {run_id_path}")
print(f"Run ID: {RUN_ID}")

---
## Summary

Here's what we accomplished:

- **Split** the training data into train (80%) and validation (20%) sets using stratified sampling
- **Built** a TF-IDF + Logistic Regression pipeline — a fast, interpretable text classifier
- **Trained** the model and evaluated it on both validation and test sets
- **Analyzed** errors using a confusion matrix and per-class classification report
- **Tracked** everything (parameters, metrics, model, plots) with MLflow

**Things to try on your own:**
- Change `C` in `configs/baseline.yaml` (try 0.1, 1.0, 10.0) and see how it affects performance
- Set `ngram_range` to `[1, 1]` (unigrams only) — how much do bigrams help?
- Reduce `max_features` to 5000 — does the model still work well?
- Compare all your runs in the MLflow UI

**Next up:** [03_serve_and_predict.ipynb](./03_serve_and_predict.ipynb) — we'll load the saved model and use it to classify new text, just like the FastAPI endpoint does.